In [ ]:
# ===== Road Sign Models — Testing & Evaluation =====
# Loads all trained models and evaluates them on clean data,
# individual attacks, and the full attack-defense matrix.

import os, sys, json
sys.path.insert(0, os.getcwd())

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from models.road_sign_classifier import (
    RoadSignClassifier, NormalizedModel, load_road_sign_classifier_checkpoint,
)
from models.target_model import DetectorNet
from road_sign_data import (
    CLASS_NAMES, load_records, stratified_split,
    RoadSignCropDataset, DisplayTensorDataset,
)
from attacks.fgsm import fgsm_attack, fgsm_attack_single
from attacks.pgd import pgd_attack, pgd_attack_single
from attacks.genetic_attack import genetic_attack
from attacks.differential_evolution_attack import de_attack
from defenses.input_transformation import apply_input_transforms
from defenses.detection_network import detect_and_predict

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# ===== Load Models & Data =====
def load_model(path, device):
    clf, ckpt = load_road_sign_classifier_checkpoint(path, device=device)
    model = NormalizedModel(clf).to(device)
    model.eval()
    return model, ckpt

models = {}
for name, path in [
    ("base", "saved_models/road_sign_crop_resnet34.pth"),
    ("adv_trained", "saved_models/road_sign_crop_adv_trained.pth"),
    ("distilled", "saved_models/road_sign_crop_distilled.pth"),
]:
    if os.path.exists(path):
        models[name], _ = load_model(path, device)
        print(f"Loaded {name}: {path}")
    else:
        print(f"MISSING {path}")

# Detector
det_path = "saved_models/road_sign_crop_detector.pth"
if os.path.exists(det_path):
    detector = DetectorNet(input_dim=512).to(device)
    detector.load_state_dict(torch.load(det_path, map_location=device, weights_only=True))
    detector.eval()
    models["detector"] = detector
    print(f"Loaded detector: {det_path}")
else:
    print(f"MISSING {det_path}")

# Validation dataset — [0,1] display images
records = load_records("annotations", "images")
_, val_records = stratified_split(records, val_ratio=0.2, seed=42)
val_ds = RoadSignCropDataset(val_records, image_size=224, augment=False, return_display=True)
val_display_ds = DisplayTensorDataset(val_ds)
test_loader = DataLoader(val_display_ds, batch_size=64, shuffle=False, num_workers=0)
print(f"Validation set: {len(val_display_ds)} images")

In [ ]:
# ===== Clean Accuracy & Confusion Matrix =====
NUM_CLASSES = len(CLASS_NAMES)

def compute_accuracy(model, loader, device):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            correct += (model(imgs).argmax(1) == lbls).sum().item()
            total += lbls.size(0)
    return 100.0 * correct / total

print("Model                  Clean Accuracy")
print("-" * 42)
acc_results = {}
for name in ["base", "adv_trained", "distilled"]:
    if name in models:
        acc = compute_accuracy(models[name], test_loader, device)
        acc_results[name] = acc
        print(f"  {name:<20} {acc:.2f}%")

# Confusion matrix for base model
def confusion_matrix(model, loader, num_classes, device):
    cm = torch.zeros(num_classes, num_classes, dtype=torch.long)
    model.eval()
    with torch.no_grad():
        for imgs, lbls in loader:
            preds = model(imgs.to(device)).argmax(1).cpu()
            for t, p in zip(lbls, preds):
                cm[t, p] += 1
    return cm.numpy()

cm = confusion_matrix(models["base"], test_loader, NUM_CLASSES, device)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
# Bar chart
colors = ["#3498db", "#e74c3c", "#2ecc71"]
bars = ax1.bar(acc_results.keys(), acc_results.values(), color=colors[:len(acc_results)], width=0.5)
ax1.set_ylim(max(0, min(acc_results.values()) - 10), 100)
ax1.set_ylabel("Accuracy (%)")
ax1.set_title("Clean Accuracy", fontweight="bold")
for b, v in zip(bars, acc_results.values()):
    ax1.text(b.get_x() + b.get_width() / 2, v + 0.3, f"{v:.1f}%", ha="center", fontweight="bold")
ax1.grid(axis="y", alpha=0.3)

# Confusion matrix
im = ax2.imshow(cm, cmap="Blues")
ax2.set_xticks(range(NUM_CLASSES)); ax2.set_yticks(range(NUM_CLASSES))
ax2.set_xticklabels(CLASS_NAMES, rotation=30, ha="right", fontsize=8)
ax2.set_yticklabels(CLASS_NAMES, fontsize=8)
ax2.set_xlabel("Predicted"); ax2.set_ylabel("True")
ax2.set_title("Base Model Confusion Matrix", fontweight="bold")
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        c = "white" if cm[i, j] > cm.max() / 2 else "black"
        ax2.text(j, i, str(cm[i, j]), ha="center", va="center", color=c, fontsize=9)
plt.colorbar(im, ax=ax2)
plt.tight_layout(); plt.show()

per_class = cm.diagonal() / cm.sum(axis=1) * 100
for i, name in enumerate(CLASS_NAMES):
    print(f"  {name}: {per_class[i]:.1f}%")

In [ ]:
# ===== Single-Image Attack Demos =====
EPSILON = 8 / 255
SAMPLE_IDX = 3

image, label = val_display_ds[SAMPLE_IDX]
print(f"Sample index: {SAMPLE_IDX}, True label: {CLASS_NAMES[label]} ({label})")

def plot_attack_result(res, title):
    fig, axes = plt.subplots(1, 4, figsize=(14, 3))
    fig.suptitle(title, fontsize=13, fontweight="bold")
    # Original
    axes[0].imshow(res["original"].permute(1, 2, 0).numpy().clip(0, 1))
    axes[0].set_title(f"Original\nPred: {CLASS_NAMES[res['orig_pred']]}")
    # Adversarial
    axes[1].imshow(res["adversarial"].permute(1, 2, 0).numpy().clip(0, 1))
    color = "red" if res["success"] else "green"
    axes[1].set_title(f"Adversarial\nPred: {CLASS_NAMES[res['adv_pred']]}", color=color)
    # Perturbation (amplified)
    pert = res["perturbation"].permute(1, 2, 0).numpy()
    axes[2].imshow((pert * 10 + 0.5).clip(0, 1))
    axes[2].set_title(f"Perturbation (10x)\nL_inf={res['l_inf']:.4f}")
    # Confidence bars
    x = range(NUM_CLASSES)
    axes[3].bar(x, res["orig_probs"], alpha=0.5, label="Original", color="#3498db")
    axes[3].bar(x, res["adv_probs"], alpha=0.5, label="Adversarial", color="#e74c3c")
    axes[3].set_xticks(x); axes[3].set_xticklabels(CLASS_NAMES, rotation=30, fontsize=7)
    axes[3].set_title("Confidence"); axes[3].legend(fontsize=8)
    for a in axes[:3]: a.axis("off")
    plt.tight_layout(); plt.show()
    status = "SUCCESS" if res["success"] else "FAILED"
    print(f"  {status} | L_inf={res['l_inf']:.4f} | L2={res['l2']:.4f}")

# FGSM
res_fgsm = fgsm_attack_single(models["base"], image, label, EPSILON, device)
plot_attack_result(res_fgsm, "FGSM Attack")

# PGD
res_pgd = pgd_attack_single(models["base"], image, label, EPSILON, steps=40, device=device)
plot_attack_result(res_pgd, "PGD Attack (40 steps)")

In [ ]:
# ===== Batch Defense Comparison (FGSM & PGD) =====
test_images, test_labels = [], []
for imgs, lbls in test_loader:
    test_images.append(imgs); test_labels.append(lbls)
    if sum(x.size(0) for x in test_images) >= 20:
        break
test_images = torch.cat(test_images)[:20].to(device)
test_labels = torch.cat(test_labels)[:20].to(device)

def robust_accuracy(model, adv_images, labels):
    model.eval()
    with torch.no_grad():
        preds = model(adv_images).argmax(1)
    return (preds == labels).float().mean().item() * 100

adv_fgsm, _, _ = fgsm_attack(models["base"], test_images, test_labels, EPSILON, device)
adv_pgd, _, _ = pgd_attack(models["base"], test_images, test_labels, EPSILON, device=device)

print(f"{'Attack':<10} {'Defense':<22} {'Robust Acc':>10}")
print("-" * 45)
defense_results = {}
for atk_name, adv_imgs in [("FGSM", adv_fgsm), ("PGD", adv_pgd)]:
    defense_results[atk_name] = {}
    for def_name, def_model in [("No Defense", models["base"]),
                                 ("Adv Training", models["adv_trained"]),
                                 ("Distillation", models["distilled"])]:
        acc = robust_accuracy(def_model, adv_imgs, test_labels)
        defense_results[atk_name][def_name] = acc
        print(f"  {atk_name:<10} {def_name:<22} {acc:>9.1f}%")

    # Input transformation
    with torch.no_grad():
        transformed = apply_input_transforms(adv_imgs)
        acc = robust_accuracy(models["base"], transformed, test_labels)
    defense_results[atk_name]["Input Transform"] = acc
    print(f"  {atk_name:<10} {'Input Transform':<22} {acc:>9.1f}%")

    # Detection
    if "detector" in models:
        preds, detected, _ = detect_and_predict(models["base"], models["detector"], adv_imgs, device)
        det_rate = detected.float().mean().item() * 100
        defense_results[atk_name]["Detection"] = det_rate
        print(f"  {atk_name:<10} {'Detection (det rate)':<22} {det_rate:>9.1f}%")
    print()

In [ ]:
# ===== Detector Score Distribution =====
if "detector" in models:
    models["base"].eval(); models["detector"].eval()
    with torch.no_grad():
        clean_feats = models["base"].get_features(test_images)
        clean_det = F.softmax(models["detector"](clean_feats), dim=1)[:, 1].cpu().numpy()
        adv_feats = models["base"].get_features(adv_pgd)
        adv_det = F.softmax(models["detector"](adv_feats), dim=1)[:, 1].cpu().numpy()

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(clean_det, bins=30, alpha=0.6, label="Clean", color="#2ecc71", edgecolor="white")
    ax.hist(adv_det, bins=30, alpha=0.6, label="Adversarial (PGD)", color="#e74c3c", edgecolor="white")
    ax.axvline(0.5, color="black", linestyle="--", label="Threshold")
    ax.set_xlabel("P(adversarial)"); ax.set_ylabel("Count")
    ax.set_title("Detection Network — Score Distribution", fontweight="bold")
    ax.legend(); plt.tight_layout(); plt.show()

    tp = (adv_det > 0.5).sum(); fn = (adv_det <= 0.5).sum()
    tn = (clean_det <= 0.5).sum(); fp = (clean_det > 0.5).sum()
    print(f"Clean  -> Detected as adversarial: {fp}/{len(clean_det)} (FPR: {fp/len(clean_det)*100:.1f}%)")
    print(f"PGD    -> Detected as adversarial: {tp}/{len(adv_det)} (TPR: {tp/len(adv_det)*100:.1f}%)")

In [ ]:
# ===== Accuracy vs Epsilon Curve =====
epsilons = [0.0, 0.01, 0.03, 0.05, 0.1]
small_batch = test_images[:15]
small_labels = test_labels[:15]

fgsm_accs, pgd_accs = [], []
for eps in epsilons:
    if eps == 0:
        acc = robust_accuracy(models["base"], small_batch, small_labels)
        fgsm_accs.append(acc); pgd_accs.append(acc)
    else:
        af, _, _ = fgsm_attack(models["base"], small_batch, small_labels, eps, device)
        fgsm_accs.append(robust_accuracy(models["base"], af, small_labels))
        ap, _, _ = pgd_attack(models["base"], small_batch, small_labels, eps, device=device)
        pgd_accs.append(robust_accuracy(models["base"], ap, small_labels))
    print(f"  eps={eps:.3f}: FGSM={fgsm_accs[-1]:.1f}%, PGD={pgd_accs[-1]:.1f}%")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(epsilons, fgsm_accs, "o-", label="FGSM", color="#e74c3c", linewidth=2)
ax.plot(epsilons, pgd_accs, "s-", label="PGD", color="#9b59b6", linewidth=2)
ax.set_xlabel("Epsilon"); ax.set_ylabel("Accuracy (%)")
ax.set_title("Base Model Accuracy vs Perturbation Budget", fontweight="bold")
ax.legend(); ax.grid(alpha=0.3); ax.set_ylim(-5, 105)
plt.tight_layout(); plt.show()

In [ ]:
# ===== Visual Defense Comparison on a Single Image =====
idx = 5
img, lbl = val_display_ds[idx]
img_dev = img.unsqueeze(0).to(device)
lbl_dev = torch.tensor([lbl], device=device)

adv, pert, _ = pgd_attack(models["base"], img_dev, lbl_dev, EPSILON, device=device)

fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))
fig.suptitle(f"PGD Attack on '{CLASS_NAMES[lbl]}' — Defense Comparison", fontweight="bold", fontsize=13)

panels = [
    ("Original", img, models["base"], img_dev),
    ("No Defense", adv.squeeze(0).cpu(), models["base"], adv),
    ("Adv Training", adv.squeeze(0).cpu(), models["adv_trained"], adv),
    ("Distillation", adv.squeeze(0).cpu(), models["distilled"], adv),
]

for i, (title, show_img, model, inp) in enumerate(panels):
    axes[i].imshow(show_img.permute(1, 2, 0).detach().numpy().clip(0, 1))
    with torch.no_grad():
        pred = model(inp.to(device)).argmax(1).item()
    color = "green" if pred == lbl else "red"
    axes[i].set_title(f"{title}\nPred: {CLASS_NAMES[pred]}", color=color, fontweight="bold")
    axes[i].axis("off")

# Input transform defense
with torch.no_grad():
    transformed = apply_input_transforms(adv)
    pred_t = models["base"](transformed).argmax(1).item()
axes[4].imshow(transformed.squeeze(0).permute(1, 2, 0).cpu().numpy().clip(0, 1))
color = "green" if pred_t == lbl else "red"
axes[4].set_title(f"Input Transform\nPred: {CLASS_NAMES[pred_t]}", color=color, fontweight="bold")
axes[4].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# ===== Confidence Distribution: Clean vs Adversarial =====
batch_imgs = test_images[:20]
batch_lbls = test_labels[:20]
adv_batch, _, _ = pgd_attack(models["base"], batch_imgs, batch_lbls, EPSILON, device=device)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, model) in zip(axes, [("Base", models["base"]),
                                     ("Adv Trained", models["adv_trained"]),
                                     ("Distilled", models["distilled"])]):
    model.eval()
    with torch.no_grad():
        clean_conf = F.softmax(model(batch_imgs), dim=1).max(1)[0].cpu().numpy()
        adv_conf = F.softmax(model(adv_batch), dim=1).max(1)[0].cpu().numpy()
    ax.hist(clean_conf, bins=15, alpha=0.6, label="Clean", color="#2ecc71", edgecolor="white")
    ax.hist(adv_conf, bins=15, alpha=0.6, label="PGD Adv", color="#e74c3c", edgecolor="white")
    ax.set_title(name, fontweight="bold")
    ax.set_xlabel("Max Confidence"); ax.legend(fontsize=9)

fig.suptitle("Confidence Distribution: Clean vs Adversarial", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# ===== Full Evaluation Heatmap (from saved results) =====
results_path = "results/evaluation_results.json"
if not os.path.exists(results_path):
    print("No evaluation results found — run train_models.ipynb Step 7 first.")
else:
    with open(results_path) as f:
        results = json.load(f)

    atk_names = results["attack_names"]
    def_names = results["defense_names"]
    matrix = results["results"]
    atk_keys = list(atk_names.keys())
    def_keys = list(def_names.keys())

    asr = np.zeros((len(atk_keys), len(def_keys)))
    for i, ak in enumerate(atk_keys):
        for j, dk in enumerate(def_keys):
            asr[i, j] = matrix.get(ak, {}).get(dk, {}).get("attack_success_rate", 0)

    fig, ax = plt.subplots(figsize=(10, 4.5))
    im = ax.imshow(asr, cmap="RdYlGn_r", aspect="auto", vmin=0, vmax=100)
    ax.set_xticks(range(len(def_keys)))
    ax.set_xticklabels([def_names[k] for k in def_keys], rotation=25, ha="right")
    ax.set_yticks(range(len(atk_keys)))
    ax.set_yticklabels([atk_names[k] for k in atk_keys])
    for i in range(len(atk_keys)):
        for j in range(len(def_keys)):
            c = "white" if asr[i, j] > 60 else "black"
            ax.text(j, i, f"{asr[i,j]:.1f}%", ha="center", va="center",
                    fontweight="bold", color=c, fontsize=10)
    ax.set_title("Attack Success Rate (%) — Lower = Better Defense",
                 fontsize=13, fontweight="bold", pad=12)
    plt.colorbar(im, ax=ax, label="ASR (%)")
    plt.tight_layout(); plt.show()

    # Print table
    print(f"\n{'Attack':<28}", end="")
    for dk in def_keys:
        print(f"{def_names[dk]:>16}", end="")
    print("\n" + "-" * 108)
    for ak in atk_keys:
        print(f"{atk_names[ak]:<28}", end="")
        for dk in def_keys:
            v = matrix.get(ak, {}).get(dk, {}).get("attack_success_rate", 0)
            print(f"{v:>15.1f}%", end="")
        print()

In [ ]:
# ===== Summary =====
print("All tests complete!")
print()
print("Saved models verified:")
for f in ["road_sign_crop_resnet34.pth", "road_sign_crop_adv_trained.pth",
          "road_sign_crop_distilled.pth", "road_sign_crop_detector.pth"]:
    path = f"saved_models/{f}"
    sz = os.path.getsize(path) / 1024 if os.path.exists(path) else 0
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"  [{status}] {path:<50} ({sz:.0f} KB)")
print()
print("To launch the web interface:")
print("  python app.py")
print("  Open: http://localhost:5000")